In [1]:
import torch
import torch.nn as nn
import torch.utils.data
import torchvision
import torchvision.transforms as transforms
import torch_directml
import matplotlib.pyplot as plt

In [13]:
# Variables
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(torch.cuda.get_device_name(0))
elif torch_directml.is_available():
   device = torch_directml.device()
   print(torch_directml.device_name(0))
else:
    device = torch.device("cpu")
    print("cpu")

epochs = 30
batch_size = 100
learning_rate = 0.001

AMD Radeon RX 6800S 


In [3]:
# Data Transformer
transformer = transforms.Compose([
    transforms.Pad(4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32),
    transforms.ToTensor(),
])

In [4]:
# Data
train_dataset = torchvision.datasets.CIFAR10(root='./source',
                                             train=True,
                                             transform=transformer,
                                             download=True)
test_dataset = torchvision.datasets.CIFAR10(root='./source',
                                            train=False,
                                            transform=transforms.ToTensor(),
                                            download=True)

Files already downloaded and verified


C:\Alexey\Projects\Udemy\NN_learning\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Files already downloaded and verified


In [5]:
# Data Loaders
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)

In [6]:
# Get conv layer
def conv_3x3(in_channels, out_channels, stride=1):
    return nn.Conv2d(in_channels,
                     out_channels,
                     kernel_size=3,
                     stride=stride,
                     padding=1,
                     bias=False)

In [7]:
# Block for ResNet
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = conv_3x3(in_channels, out_channels, stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv_3x3(out_channels, out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
    def forward(self, x):
        residual = x.clone()
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

In [8]:
# Network class
class ResNet(nn.Module):
    def __init__(self, block, num_classes=10):
        super().__init__()
        self.in_channels = 16
        self.conv = conv_3x3(in_channels=3,
                            out_channels=16,
                            stride=1)
        self.bn = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self.make_layer(block, 16, stride=1)
        self.layer2 = self.make_layer(block, 32, stride=2)
        self.layer3 = self.make_layer(block, 64, stride=2)
        self.avg_pool = nn.AvgPool2d(kernel_size=8)
        self.fc = nn.Linear(64, num_classes)
    def make_layer(self, block, out_channels, stride=1):
        downsample = None
        if self.in_channels != out_channels or stride != 1:
            downsample = nn.Sequential(
                conv_3x3(self.in_channels, out_channels, stride),
                nn.BatchNorm2d(out_channels)
            )
        res_blocks = list()
        res_blocks.append(block(in_channels=self.in_channels,
                                        out_channels=out_channels,
                                        stride=stride,
                                        downsample=downsample))
        self.in_channels = out_channels
        res_blocks.append(block(in_channels=self.in_channels,
                                out_channels=out_channels))
        return nn.Sequential(*res_blocks)
    def forward(self, x):
        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avg_pool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [9]:
# Modelling
model = ResNet(block = ResidualBlock,
               num_classes=10)
model = model.to(device)
criterion = nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.SGD(params=model.parameters(),
                            lr=learning_rate,
                            momentum=0.9,
                            weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

In [14]:
# Learning
for epoch in range(epochs):
    model.train()
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        _, predictions = torch.max(outputs, 1)
        if (i + 1) % 100 == 0:
            print(f"Epoch {epoch}/{epochs}")
            print(f"Step {i + 1}/{len(train_loader)}")
            print(f"Loss: {loss.item():.4f}")
            print(f"Accuracy: {(predictions == labels).sum() / len(labels) * 100:.4f}%")
            print()
    scheduler.step()

Epoch 0/30
Step 100/500
Loss: 0.6479
Accuracy: 77.0000%

Epoch 0/30
Step 200/500
Loss: 0.6449
Accuracy: 73.0000%

Epoch 0/30
Step 300/500
Loss: 0.7063
Accuracy: 80.0000%

Epoch 0/30
Step 400/500
Loss: 0.7691
Accuracy: 73.0000%

Epoch 0/30
Step 500/500
Loss: 0.7318
Accuracy: 77.0000%

Epoch 1/30
Step 100/500
Loss: 0.7156
Accuracy: 72.0000%

Epoch 1/30
Step 200/500
Loss: 0.5736
Accuracy: 79.0000%

Epoch 1/30
Step 300/500
Loss: 0.6073
Accuracy: 79.0000%

Epoch 1/30
Step 400/500
Loss: 0.6613
Accuracy: 78.0000%

Epoch 1/30
Step 500/500
Loss: 0.7037
Accuracy: 73.0000%

Epoch 2/30
Step 100/500
Loss: 0.7268
Accuracy: 76.0000%

Epoch 2/30
Step 200/500
Loss: 0.4560
Accuracy: 85.0000%

Epoch 2/30
Step 300/500
Loss: 0.5237
Accuracy: 82.0000%

Epoch 2/30
Step 400/500
Loss: 0.4601
Accuracy: 85.0000%

Epoch 2/30
Step 500/500
Loss: 0.4424
Accuracy: 81.0000%

Epoch 3/30
Step 100/500
Loss: 0.6284
Accuracy: 78.0000%

Epoch 3/30
Step 200/500
Loss: 0.4920
Accuracy: 83.0000%

Epoch 3/30
Step 300/500
Loss: 0

In [15]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        total += len(labels)
        correct += (predictions == labels).sum().item()
print(f"Total Accuracy: {correct / total * 100}")

Total Accuracy: 86.85000000000001
